Pruebo si la sesión de spark crea Data frame

In [1]:
import os
os.environ["JAVA_HOME"] = "C:\\Program Files\\Eclipse Adoptium\\jdk-17.0.18.8-hotspot"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

print("🔥 Spark arrancó correctamente")

df = spark.createDataFrame([{"a": 1}, {"a": 2}])

df.show()   # 👈 ESTO FALTABA

spark.stop()

🔥 Spark arrancó correctamente
+---+
|  a|
+---+
|  1|
|  2|
+---+



Pruebo test_mfcc

probando con funciones separadas

In [7]:
#codigo de prueba sin trasformar a mfcc
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, FloatType


import librosa

In [8]:
# 🔹 Función que transforma UN audio → MFCC
def extract_mfcc(path):
    try:
        # cargar audio
        y, sr = librosa.load(path, sr=22050)

        # extraer MFCC
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

        # convertir a lista (Spark no guarda numpy directo)
        return mfcc.flatten().tolist()

    except Exception as e:
        print(f"❌ Error en {path}: {e}")
        return []

In [ ]:
def process_mfcc(data):

    # 1. Crear e iniciar sesión de Spark
    spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Audio MFCC Pipeline") \
    .getOrCreate()

    # 👈 ESTA ES LA CLAVE: Envía el archivo al clúster de Spark
    # Esto permite que los workers encuentren el módulo 'etl'
    #spark.sparkContext.addPyFile("etl/mfcc_pyspark.py")

    # 2. Realizar operaciones con la sesión de Spark
    # Convertir lista Python → DataFrame Spark    
    df = spark.createDataFrame(data)

    # 3. Registrar función como UDF (User Defined Function)
    mfcc_udf = udf(extract_mfcc, ArrayType(FloatType()))

    # 4. Aplicar transformación (columna nueva)
    df = df.withColumn("mfcc", mfcc_udf(df["path"]))

    print("✅ MFCC generados con PySpark")
    
    #df.printSchema()
    df.show(5, truncate=False)

    
    return df

In [13]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from etl.ingest import ingest_data
#from etl.mfcc_pyspark import process_mfcc

data = ingest_data("..\\data\\raw")
df = process_mfcc(data)

df.show(5)


✅ Total audios cargados: 20
✅ MFCC generados con PySpark
+----------+-----+-----------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Py4JJavaError: An error occurred while calling o226.showString.
: java.lang.IllegalStateException: SparkContext has been shutdown
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2390)
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2419)
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2438)
	at org.apache.spark.sql.execution.SparkPlan.executeTake(SparkPlan.scala:530)
	at org.apache.spark.sql.execution.SparkPlan.executeTake(SparkPlan.scala:483)
	at org.apache.spark.sql.execution.CollectLimitExec.executeCollect(limit.scala:61)
	at org.apache.spark.sql.Dataset.collectFromPlan(Dataset.scala:4332)
	at org.apache.spark.sql.Dataset.$anonfun$head$1(Dataset.scala:3314)
	at org.apache.spark.sql.Dataset.$anonfun$withAction$2(Dataset.scala:4322)
	at org.apache.spark.sql.execution.QueryExecution$.withInternalError(QueryExecution.scala:546)
	at org.apache.spark.sql.Dataset.$anonfun$withAction$1(Dataset.scala:4320)
	at org.apache.spark.sql.execution.SQL